# Marked-obstruction object completion test

Use two same-size images: (1) the untouched photo and (2) a duplicate where you paint the obstruction bright magenta (#FF00FF). The notebook extracts that painted region as the inpainting mask, runs LaMa, then composites the prediction only inside the mask.

This is the user-guided route. LaMa fills the marked pixels plausibly; it cannot know the exact hidden chair/object details. Use a photo where the obstruction itself covers the target object.

Run cells in order. A free Colab GPU is optional for trying the workflow; the notebook selects CUDA if available.

In [ ]:
%pip -q install simple-lama-inpainting pillow numpy
import numpy as np
from PIL import Image, ImageFilter
from IPython.display import display
import torch
from simple_lama_inpainting import SimpleLama
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## Upload both images

Paint the obstruction on the duplicate using solid bright magenta. Keep the canvas dimensions unchanged and do not paint the object itself unless it is also the obstruction.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
if len(uploaded) != 2:
    raise ValueError("Upload exactly two files: original photo and magenta-marked duplicate.")
names = list(uploaded)
original_name = next((n for n in names if "marked" not in n.lower() and "mask" not in n.lower()), names[0])
marked_name = next((n for n in names if n != original_name), names[1])
original = Image.open(original_name).convert("RGB")
marked = Image.open(marked_name).convert("RGB")
if original.size != marked.size:
    raise ValueError(f"Images must be the same size. Original={original.size}, marked={marked.size}")
display(original, marked)
print("Original:", original_name, "| Marked copy:", marked_name)


## Extract the mark and run LaMa

The white preview is the exact region sent to the model. If it does not match the obstruction, stop and fix the marked copy before running inference.

In [ ]:
src = np.asarray(original, dtype=np.int16)
ann = np.asarray(marked, dtype=np.int16)
is_magenta = (ann[...,0] >= 170) & (ann[...,2] >= 170) & (ann[...,1] <= 120)
was_changed = np.max(np.abs(ann-src), axis=2) >= 35
mask = Image.fromarray(((is_magenta & was_changed).astype(np.uint8)*255), mode="L").filter(ImageFilter.MaxFilter(41))
mask_array = np.asarray(mask)
if not np.any(mask_array):
    raise ValueError("No magenta marks detected. Mark the obstruction #FF00FF and upload both images again.")
display(mask)
lama = SimpleLama()
scale = min(1.0, 2048 / max(original.size))
if scale < 1.0:
    small_size = (round(original.width*scale), round(original.height*scale))
    small_image = original.resize(small_size, Image.Resampling.LANCZOS)
    small_mask = mask.resize(small_size, Image.Resampling.NEAREST)
else:
    small_image, small_mask = original, mask
prediction = lama(small_image, small_mask).convert("RGB")
if prediction.size != original.size:
    prediction = prediction.resize(original.size, Image.Resampling.LANCZOS)
completed = Image.composite(prediction, original, mask)
display(completed)
completed.save("completed_object.png")
mask.save("extracted_obstruction_mask.png")
print("Saved completed_object.png and extracted_obstruction_mask.png")


In [ ]:
files.download("completed_object.png")
